In [6]:
import numpy as np
import tensorflow as tf
#from tensorflow.keras import layers, models
from keras import layers, models
import os

# --- 1. CONFIGURAZIONE SPLIT ---
# Indici forniti per training e validation
train_indices = [22, 0, 1, 2, 3, 10, 14, 16, 17, 18, 19, 21, 8, 9, 12, 5, 6, 4]
val_indices = [23, 7, 11, 13, 15, 20]

def load_data_from_windows(indices, base_path="dataset/data/"):
    x_data = []
    y_data = []
    
    for idx in indices:
        file_path = os.path.join(base_path, f"window_{idx:06d}.npz")
        if os.path.exists(file_path):
            data = np.load(file_path)
            # Carichiamo i dati grezzi e le label (coordinate x,y)
            x_data.append(data['radar_cir_iq'])  # Shape: (T, 6, 3, 120, 2)
            y_data.append(data['people_xy']) # Shape: (T, 4, 2) - coordinate x,y per 4 persone
            
    return x_data, y_data

# --- 2. PRE-PROCESSING (Magnitude + EMA Decluttering) ---
def preprocess_batch(x_list, alpha=0.05):
    processed_x = []
    for window in x_list:
        # Calcolo Magnitudo: sqrt(I^2 + Q^2) -> Shape: (T, 6, 3, 120)
        mag = np.sqrt(window[..., 0]**2 + window[..., 1]**2)
        
        # EMA Decluttering (rimozione riflessi statici come da slide)
        bg = np.zeros_like(mag[0])
        decluttered = []
        for frame in mag:
            bg = alpha * frame + (1 - alpha) * bg
            decluttered.append(frame - bg)
            
        # Reshape per la CNN: uniamo radar e antenne come canali o feature
        # Qui trattiamo ogni frame come (120 range_bins, 18 feature) 
        # (6 radar * 3 antenne = 18 canali di informazione)
        frame_data = np.array(decluttered).transpose(0, 3, 1, 2).reshape(-1, 120, 18)
        processed_x.append(frame_data)
        
    return np.vstack(processed_x)

# --- 3. DEFINIZIONE DELLA RETE (Stile Lecture 5 CNN) ---
def build_model(input_shape):
    model = models.Sequential([
        # Input Layer: (Range Bins, Features/Channels)
        layers.Input(shape=input_shape),
        
        # Primo Blocco Convoluzionale (Conv2D richiede 3D, usiamo Conv1D per segnali 1D)
        # O usiamo Conv2D aggiungendo una dimensione fittizia per simulare un'immagine
        layers.Reshape((input_shape[0], input_shape[1], 1)),
        
        layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        # Global Average Pooling per risparmiare SRAM (visto nei Lab per Edge AI)
        layers.GlobalAveragePooling2D(),
        
        # Parte Dense per la regressione delle coordinate
        layers.Dense(64, activation='relu'),
        
        # Output: 8 valori (x1, y1, x2, y2, x3, y3, x4, y4)
        layers.Dense(8, activation='linear') 
    ])
    
    # Loss: Mean Absolute Error (MAE) o MSE, tipico per la regressione nei Lab
    model.compile(optimizer='adam', loss='mae', metrics=['mse'])
    return model

# --- ESECUZIONE ---
# 1. Caricamento
x_train_raw, y_train_raw = load_data_from_windows(train_indices)
x_val_raw, y_val_raw = load_data_from_windows(val_indices)

# 2. Pre-processing
x_train = preprocess_batch(x_train_raw)
y_train = np.vstack(y_train_raw).reshape(-1, 8) # Flatten coordinate

x_val = preprocess_batch(x_val_raw)
y_val = np.vstack(y_val_raw).reshape(-1, 8)

# 3. Creazione Modello
# Ogni input è (120 range bins, 18 feature)
input_dim = (120, 18)
model = build_model(input_dim)

model.summary()

# --- 4. VALUTAZIONE VINCOLI (Stile Lab 4/5) ---
# Calcolo parametri e stima memoria Flash
total_params = model.count_params()
estimated_flash_kb = (total_params * 4) / 1024  # Assumendo float32 (4 bytes)
print(f"\nParametri Totali: {total_params}")
print(f"Stima Flash (float32): {estimated_flash_kb:.2f} KB")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 120, 18, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 18, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 60, 9, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 60, 9, 32)      │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 4, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,432 (29.03 KB)

 Trainable params: 7,432 (29.03 KB)

 Non-trainable params: 0 (0.00 B)


Parametri Totali: 7432
Stima Flash (float32): 29.03 KB


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models, losses

def build_eeai_model(n_radars=6, n_antennas=3, n_bins=120):
    # Calcoliamo i canali totali (6 * 3 = 18 per la baseline)
    input_channels = n_radars * n_antennas
    
    # --- INPUT LAYER ---
    # Trattiamo i dati come un'immagine 1D (1 x 120) con molti canali
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- FEATURE EXTRACTION (MobileNet Style) ---
    # 1. Prima convoluzione standard per estrarre i segnali base
    x = layers.Conv2D(16, (1, 5), padding='same', activation='relu', name="conv_base")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # Riduce a 60 bin

    # 2. Separable Convolution (Risparmio RAM critico per ESP32)
    x = layers.SeparableConv2D(32, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Riduce a 30 bin

    # 3. Global Average Pooling (Distrugge l'asse spaziale, salva la memoria)
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # 4. Layer denso comune (Rappresentazione astratta della stanza)
    common_feat = layers.Dense(32, activation='relu', name="features")(x)

    # --- MULTI-HEAD OUTPUT ---
    # Testa A: Coordinate (4 persone * 2 coord = 8 output)
    # Usiamo 'linear' perché le coordinate X,Y possono essere qualsiasi valore in metri
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)

    # Testa B: Presenza (4 persone = 4 output)
    # Usiamo 'sigmoid' per avere una probabilità tra 0 e 1 per ogni slot
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    model = models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_v1")
    return model

# Costruiamo il modello
model = build_eeai_model()

# --- DEFINIZIONE DELLA LOSS PERSONALIZZATA ---
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE) ma lo annulla 
    se la persona non è presente nella ground truth.
    """
    # Assumiamo che la maschera sia passata esternamente o gestita via pesi
    # Per semplicità qui usiamo una versione standard, 
    # ma in fase di training useremo model.compile(loss_weights=...)
    return losses.mean_squared_error(y_true, y_pred)

model.compile(
    optimizer='adam',
    loss={
        "coords_head": "mse",       # Errore quadratico per la posizione
        "mask_head": "binary_crossentropy" # Cross-entropy per la presenza (0/1)
    },
    loss_weights={
        "coords_head": 1.0, 
        "mask_head": 0.5            # Diamo più importanza alla precisione della posizione
    }
)

model.summary()

Model: "EEAI_Net_v1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ radar_input         │ (None, 1, 120,    │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_base (Conv2D)  │ (None, 1, 120,    │      1,456 │ radar_input[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 1, 60, 16) │          0 │ conv_base[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sep_conv_1          │ (None, 1, 60, 32) │        592 │ pool_1[0][0]      │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 1, 30, 32) │          0 │ sep_conv_1[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 32)        │          0 │ pool_2[0][0]      │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features (Dense)    │ (None, 32)        │      1,056 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coords_head (Dense) │ (None, 8)         │        264 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_head (Dense)   │ (None, 4)         │        132 │ features[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,500 (13.67 KB)

 Trainable params: 3,500 (13.67 KB)

 Non-trainable params: 0 (0.00 B)